# E-commerce Delivery Intelligence

## Business Analysis

This notebook analyses delivery performance across the Olist e-commerce dataset, identifies patterns associated with late delivery and examines the relationship between delivery performance and customer experience.

In [1]:
import pandas as pd
import plotly.express as px

In [2]:
master = pd.read_csv(
    "../data/processed/master_orders.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

In [3]:
master.shape

(96470, 30)

In [4]:
master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,actual_delivery_days,...,primary_product_category,primary_seller_state,number_of_categories,number_of_sellers,primary_payment_type,review_score,number_of_reviews,total_payment_value,max_payment_installments,number_of_payment_records
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.436574,...,housewares,SP,1,1,voucher,4.0,1.0,38.71,1.0,3.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,13.782037,...,perfumery,SP,1,1,boleto,4.0,1.0,141.46,1.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9.394213,...,auto,SP,1,1,credit_card,5.0,1.0,179.12,3.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,13.208750,...,pet_shop,MG,1,1,credit_card,5.0,1.0,72.20,1.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,2.873877,...,stationery,SP,1,1,credit_card,5.0,1.0,28.62,1.0,1.0


In [5]:
master["order_id"].is_unique

True

In [6]:
print("Rows:", len(master))
print("Unique orders:", master["order_id"].nunique())
print("Duplicate order IDs:", master["order_id"].duplicated().sum())

Rows: 96470
Unique orders: 96470
Duplicate order IDs: 0


In [7]:
master[
    [
        "order_value",
        "freight_value",
        "number_of_items",
        "promised_delivery_days",
        "actual_delivery_days",
        "review_score"
    ]
].describe()

,order_value,freight_value,number_of_items,promised_delivery_days,actual_delivery_days,review_score
count,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,95824.000000
mean,137.040001,22.785798,1.142210,23.736343,12.558217,4.156158
std,209.052608,21.559959,0.538824,8.761052,9.546156,1.283615
min,0.850000,0.000000,1.000000,2.008009,0.533414,1.000000
25%,45.900000,13.850000,1.000000,18.329905,6.766204,4.000000
50%,86.500000,17.170000,1.000000,23.230880,10.217477,5.000000
75%,149.900000,24.020000,1.000000,28.407795,15.720182,5.000000
max,13440.000000,1794.960000,21.000000,155.135463,209.628611,5.000000


In [9]:
total_orders = len(master)

total_order_value = master["order_value"].sum()

average_order_value = master["order_value"].mean()

late_delivery_rate = master["is_late"].mean() * 100

average_review_score = master["review_score"].mean()

In [10]:
print(f"Total Orders: {total_orders:,}")
print(f"Total Order Value: R${total_order_value:,.2f}")
print(f"Average Order Value: R${average_order_value:,.2f}")
print(f"Late Delivery Rate: {late_delivery_rate:.2f}%")
print(f"Average Review Score: {average_review_score:.2f}/5")

Total Orders: 96,470
Total Order Value: R$13,220,248.93
Average Order Value: R$137.04
Late Delivery Rate: 8.11%
Average Review Score: 4.16/5


In [11]:
monthly_summary = (
    master
    .groupby("purchase_year_month")
    .agg(
        orders=("order_id", "count"),
        order_value=("order_value", "sum")
    )
    .reset_index()
)

In [12]:
monthly_summary.head()

,purchase_year_month,orders,order_value
0,2016-09,1,134.97
1,2016-10,265,40325.11
2,2016-12,1,10.90
3,2017-01,750,111798.36
4,2017-02,1653,234223.40


In [13]:
fig = px.line(
    monthly_summary,
    x="purchase_year_month",
    y="orders",
    markers=True,
    title="Monthly Order Volume"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Orders"
)

fig.show()

In [14]:
fig = px.line(
    monthly_summary,
    x="purchase_year_month",
    y="order_value",
    markers=True,
    title="Monthly Merchandise Value"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Order Value (R$)"
)

fig.show()

In [15]:
monthly_delivery = (
    master
    .groupby("purchase_year_month")
    .agg(
        orders=("order_id", "count"),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

monthly_delivery["late_delivery_rate"] *= 100

In [16]:
fig = px.line(
    monthly_delivery,
    x="purchase_year_month",
    y="late_delivery_rate",
    markers=True,
    title="Late Delivery Rate Over Time"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Late Delivery Rate (%)"
)

fig.show()

In [17]:
state_delivery = (
    master
    .groupby("customer_state")
    .agg(
        orders=("order_id", "count"),
        late_delivery_rate=("is_late", "mean"),
        average_delivery_days=("actual_delivery_days", "mean")
    )
    .reset_index()
)

state_delivery["late_delivery_rate"] *= 100

In [18]:
state_delivery.sort_values(
    "late_delivery_rate",
    ascending=False
)

,customer_state,orders,late_delivery_rate,average_delivery_days
1,AL,397,23.929471,24.543855
9,MA,717,19.665272,21.572976
16,PI,476,15.966387,19.457098
5,CE,1279,15.324472,21.266579
24,SE,335,15.223881,21.519788
4,BA,3256,14.035627,19.335466
18,RJ,12350,13.473684,15.309438
26,TO,274,12.773723,17.658063
13,PA,946,12.367865,23.772917
7,ES,1995,12.230576,15.789307


In [19]:
state_delivery_filtered = state_delivery[
    state_delivery["orders"] >= 100
].copy()

In [20]:
fig = px.bar(
    state_delivery_filtered.sort_values(
        "late_delivery_rate",
        ascending=False
    ),
    x="customer_state",
    y="late_delivery_rate",
    hover_data=["orders", "average_delivery_days"],
    title="Late Delivery Rate by Customer State"
)

fig.update_layout(
    xaxis_title="Customer State",
    yaxis_title="Late Delivery Rate (%)"
)

fig.show()

In [21]:
master["same_state_delivery"] = (
    master["customer_state"] ==
    master["primary_seller_state"]
)

In [22]:
same_state_summary = (
    master
    .groupby("same_state_delivery")
    .agg(
        orders=("order_id", "count"),
        late_delivery_rate=("is_late", "mean"),
        average_delivery_days=("actual_delivery_days", "mean")
    )
)

same_state_summary["late_delivery_rate"] *= 100

same_state_summary

,orders,late_delivery_rate,average_delivery_days
same_state_delivery,,,
False,61774,9.264415,15.149601
True,34696,6.061217,7.944424


In [23]:
category_delivery = (
    master
    .groupby("primary_product_category")
    .agg(
        orders=("order_id", "count"),
        late_delivery_rate=("is_late", "mean"),
        average_order_value=("order_value", "mean")
    )
    .reset_index()
)

category_delivery["late_delivery_rate"] *= 100

In [24]:
category_delivery_filtered = category_delivery[
    category_delivery["orders"] >= 300
].copy()

In [25]:
top_late_categories = (
    category_delivery_filtered
    .sort_values("late_delivery_rate", ascending=False)
    .head(15)
)

In [26]:
fig = px.bar(
    top_late_categories,
    x="late_delivery_rate",
    y="primary_product_category",
    orientation="h",
    hover_data=["orders", "average_order_value"],
    title="Product Categories with Highest Late Delivery Rates"
)

fig.update_layout(
    xaxis_title="Late Delivery Rate (%)",
    yaxis_title="Product Category"
)

fig.show()

In [27]:
fig.update_yaxes(categoryorder="total ascending")

In [28]:
master["order_value_group"] = pd.qcut(
    master["order_value"],
    q=5,
    labels=[
        "Lowest",
        "Low",
        "Medium",
        "High",
        "Highest"
    ]
)

In [30]:
value_delivery = (
    master
    .groupby(
        "order_value_group",
        observed=True
    )
    .agg(
        orders=("order_id", "count"),
        average_order_value=("order_value", "mean"),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

value_delivery["late_delivery_rate"] *= 100

value_delivery

,order_value_group,orders,average_order_value,late_delivery_rate
0,Lowest,19452,25.218770,7.289739
1,Low,19178,52.328834,7.957034
2,Medium,19253,86.024202,7.936425
3,High,19298,135.572769,8.617473
4,Highest,19289,386.418362,8.766655


In [31]:
fig = px.bar(
    value_delivery,
    x="order_value_group",
    y="late_delivery_rate",
    title="Late Delivery Rate by Order Value"
)

fig.update_layout(
    xaxis_title="Order Value Group",
    yaxis_title="Late Delivery Rate (%)"
)

fig.show()

In [32]:
master["freight_group"] = pd.qcut(
    master["freight_value"],
    q=5,
    labels=[
        "Lowest",
        "Low",
        "Medium",
        "High",
        "Highest"
    ],
    duplicates="drop"
)

In [33]:
freight_delivery = (
    master
    .groupby(
        "freight_group",
        observed=True
    )
    .agg(
        orders=("order_id", "count"),
        average_freight=("freight_value", "mean"),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

freight_delivery["late_delivery_rate"] *= 100

freight_delivery

,freight_group,orders,average_freight,late_delivery_rate
0,Lowest,19397,9.745603,6.078260
1,Low,19192,14.475345,7.237391
2,Medium,19380,17.187369,8.859649
3,High,19225,22.232285,9.045514
4,Highest,19276,50.362768,9.348413


In [34]:
fig = px.bar(
    freight_delivery,
    x="freight_group",
    y="late_delivery_rate",
    title="Late Delivery Rate by Freight Cost"
)

fig.update_layout(
    xaxis_title="Freight Cost Group",
    yaxis_title="Late Delivery Rate (%)"
)

fig.show()

In [35]:
master["number_of_items"].value_counts().sort_index().head(15)

number_of_items
1     86835
2      7392
3      1306
4       495
5       193
6       191
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
Name: count, dtype: int64

In [36]:
item_count_delivery = (
    master
    .groupby("number_of_items")
    .agg(
        orders=("order_id", "count"),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

item_count_delivery["late_delivery_rate"] *= 100

In [37]:
item_count_chart = item_count_delivery[
    item_count_delivery["number_of_items"] <= 5
]

In [38]:
fig = px.bar(
    item_count_chart,
    x="number_of_items",
    y="late_delivery_rate",
    title="Late Delivery Rate by Number of Items"
)

fig.update_layout(
    xaxis_title="Items in Order",
    yaxis_title="Late Delivery Rate (%)"
)

fig.show()

In [39]:
seller_complexity = (
    master
    .groupby("number_of_sellers")
    .agg(
        orders=("order_id", "count"),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

seller_complexity["late_delivery_rate"] *= 100

seller_complexity

,number_of_sellers,orders,late_delivery_rate
0,1,95195,8.202111
1,2,1216,1.480263
2,3,54,0.000000
3,4,3,0.000000
4,5,2,0.000000


In [40]:
master["promised_delivery_days"].describe()

count    96470.000000
mean        23.736343
std          8.761052
min          2.008009
25%         18.329905
50%         23.230880
75%         28.407795
max        155.135463
Name: promised_delivery_days, dtype: float64

In [41]:
master["promise_window_group"] = pd.qcut(
    master["promised_delivery_days"],
    q=5,
    labels=[
        "Shortest",
        "Short",
        "Medium",
        "Long",
        "Longest"
    ]
)

In [42]:
promise_delivery = (
    master
    .groupby(
        "promise_window_group",
        observed=True
    )
    .agg(
        orders=("order_id", "count"),
        average_promised_days=(
            "promised_delivery_days",
            "mean"
        ),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

promise_delivery["late_delivery_rate"] *= 100

promise_delivery

,promise_window_group,orders,average_promised_days,late_delivery_rate
0,Shortest,19294,12.402127,8.557064
1,Short,19294,19.402755,9.655852
2,Medium,19294,23.180334,8.816212
3,Long,19294,27.302304,8.126879
4,Longest,19294,36.394194,5.405826


In [43]:
fig = px.bar(
    promise_delivery,
    x="promise_window_group",
    y="late_delivery_rate",
    title="Late Delivery Rate by Promised Delivery Window"
)

fig.update_layout(
    xaxis_title="Promised Delivery Window",
    yaxis_title="Late Delivery Rate (%)"
)

fig.show()

In [44]:
review_delivery = (
    master
    .groupby("is_late")
    .agg(
        orders=("order_id", "count"),
        average_review_score=("review_score", "mean")
    )
    .reset_index()
)

review_delivery

,is_late,orders,average_review_score
0,0,88644,4.294292
1,1,7826,2.566506


In [45]:
review_delivery["delivery_status"] = (
    review_delivery["is_late"]
    .map({
        0: "On Time",
        1: "Late"
    })
)

In [46]:
fig = px.bar(
    review_delivery,
    x="delivery_status",
    y="average_review_score",
    title="Average Review Score: On-Time vs Late Orders"
)

fig.update_layout(
    xaxis_title="Delivery Status",
    yaxis_title="Average Review Score"
)

fig.show()

In [47]:
review_distribution = (
    master
    .groupby(["is_late", "review_score"])
    .size()
    .reset_index(name="orders")
)

In [48]:
review_distribution["percentage"] = (
    review_distribution
    .groupby("is_late")["orders"]
    .transform(lambda x: x / x.sum() * 100)
)

In [49]:
review_distribution["delivery_status"] = (
    review_distribution["is_late"]
    .map({
        0: "On Time",
        1: "Late"
    })
)

In [50]:
fig = px.bar(
    review_distribution,
    x="review_score",
    y="percentage",
    color="delivery_status",
    barmode="group",
    title="Review Score Distribution by Delivery Status"
)

fig.update_layout(
    xaxis_title="Review Score",
    yaxis_title="Percentage of Orders"
)

fig.show()

In [51]:
driver_summary = pd.DataFrame({
    "Metric": [
        "Late Delivery Rate",
        "Average Review - On Time",
        "Average Review - Late",
        "Average Delivery Days",
        "Average Promised Days"
    ],
    "Value": [
        master["is_late"].mean() * 100,
        master.loc[
            master["is_late"] == 0,
            "review_score"
        ].mean(),
        master.loc[
            master["is_late"] == 1,
            "review_score"
        ].mean(),
        master["actual_delivery_days"].mean(),
        master["promised_delivery_days"].mean()
    ]
})

driver_summary

,Metric,Value
0,Late Delivery Rate,8.112367
1,Average Review - On Time,4.294292
2,Average Review - Late,2.566506
3,Average Delivery Days,12.558217
4,Average Promised Days,23.736343


## Preliminary Business Findings

## Preliminary Business Findings

### Finding 1: Late delivery is strongly associated with poorer customer experience

Late orders received an average review score of **2.57/5**, compared with **4.29/5** for orders delivered on time. This 1.72-point difference indicates a substantial relationship between delivery performance and customer satisfaction.

From a business perspective, reducing late deliveries may therefore have benefits beyond logistics performance by protecting the overall customer experience.

### Finding 2: Delivery performance varies substantially by customer geography

The overall late-delivery rate was **8.11%**, but several customer states experienced considerably higher rates. Among states with at least 100 orders, **Alagoas (AL)** recorded a late-delivery rate of **23.93%**, followed by **Maranhão (MA)** at **19.67%**.

This suggests that delivery risk is not evenly distributed geographically and that particular regions may warrant further investigation into logistics constraints or fulfilment performance.

### Finding 3: Higher freight costs are associated with greater delivery risk

Orders in the lowest freight-cost group had a late-delivery rate of **6.08%**, compared with **9.35%** for orders in the highest freight-cost group.

The pattern is broadly increasing across freight-cost groups, suggesting that orders involving higher freight costs may also face greater logistical complexity or distance. This relationship is associative rather than causal.

### Finding 4: Longer promised delivery windows are associated with lower late-delivery rates

Orders with the longest promised delivery windows had a late-delivery rate of **5.41%**, compared with rates of approximately **8–10%** across shorter delivery-window groups.

This suggests that additional delivery buffer may be associated with improved on-time performance, although other factors such as geography, product type and shipping complexity may also explain part of the relationship.

### Finding 5: Certain product categories experience elevated delivery risk

Among product categories with at least 300 orders, **audio** products recorded the highest late-delivery rate at **13.08%**, followed by **home comfort** at **10.73%** and **food** at **10.11%**.

Category-level differences may indicate varying fulfilment requirements, seller characteristics or logistics complexity and warrant further investigation.